# Exam Paper Generator
Loads SQuAD and RACE from local files. Extracts key concepts using NER and generates MCQs, short answers, and descriptive questions using T5.

## Step 1: Install Dependencies

In [21]:
# Install all required packages
!pip install transformers torch sentencepiece rouge-score spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.5 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Step 2: Imports and Configuration

In [22]:
# Standard library imports
import os
import json
import glob
import random

# Deep learning and NLP imports
import torch
import spacy
from transformers import T5ForConditionalGeneration, T5Tokenizer
from rouge_score import rouge_scorer

# Load spacy NER model
nlpModel = spacy.load('en_core_web_sm')

# Device configuration - uses GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Step 3: Configure Dataset Paths
Update these paths to match where your datasets are stored on your machine.

In [23]:
# --- UPDATE THESE PATHS TO MATCH YOUR LOCAL SETUP ---

# Path to the RACE_questions folder (contains dev/, test/, train/ subfolders)
raceFolderPath = '/kaggle/input/datasets/anandikam/race-d/RACE'

# Paths to SQuAD JSON files
squadTrainPath = '/kaggle/input/datasets/organizations/stanfordu/stanford-question-answering-dataset/train-v1.1.json'
squadDevPath   = '/kaggle/input/datasets/organizations/stanfordu/stanford-question-answering-dataset/dev-v1.1.json'

# --- END OF PATH CONFIGURATION ---

## Step 4: Load SQuAD from Local JSON

In [24]:
# Parse a SQuAD v1.1 JSON file into a flat list of context/question/answer dicts
def parseSquadFile(filePath):
    # Returns a flat list of samples, each with context, question, and answers
    with open(filePath, 'r', encoding='utf-8') as f:
        rawData = json.load(f)
    samples = []
    for article in rawData['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            for qa in paragraph['qas']:
                samples.append({
                    'context': context,
                    'question': qa['question'],
                    'answers': qa['answers']
                })
    return samples

# Load both train and dev splits of SQuAD
def loadSquadLocal(trainPath, devPath):
    # Returns a dict with train and validation sample lists
    trainSamples = parseSquadFile(trainPath)
    devSamples = parseSquadFile(devPath)
    print(f'SQuAD train size: {len(trainSamples)}')
    print(f'SQuAD validation size: {len(devSamples)}')
    return {'train': trainSamples, 'validation': devSamples}

squadData = loadSquadLocal(squadTrainPath, squadDevPath)

SQuAD train size: 87599
SQuAD validation size: 10570


## Step 5: Load RACE from Local Folder

In [25]:
# Parse a single RACE .txt JSON file into a sample dict
def parseRaceFile(filePath):
    # Returns a list of samples from one RACE file
    with open(filePath, 'r', encoding='utf-8') as f:
        rawData = json.load(f)
    samples = []
    article = rawData.get('article', '')
    questions = rawData.get('questions', [])
    optionsList = rawData.get('options', [])
    answers = rawData.get('answers', [])
    for i, question in enumerate(questions):
        samples.append({
            'article': article,
            'question': question,
            'options': optionsList[i],
            'answer': answers[i]
        })
    return samples

# Load all RACE files from a given split folder (train / dev / test)
def loadRaceSplit(splitFolderPath):
    # Returns a flat list of all samples from both high and middle difficulty folders
    highFiles = glob.glob(os.path.join(splitFolderPath, 'high', '*.txt'))
    middleFiles = glob.glob(os.path.join(splitFolderPath, 'middle', '*.txt'))
    allFiles = highFiles + middleFiles
    allSamples = []
    for filePath in allFiles:
        try:
            fileSamples = parseRaceFile(filePath)
            allSamples.extend(fileSamples)
        except Exception as err:
            print(f'Skipping file {filePath}: {err}')
    return allSamples

# Load train and dev splits of RACE from local folder
def loadRaceLocal(raceFolderPath):
    # Returns a dict with train and validation sample lists
    trainSamples = loadRaceSplit(os.path.join(raceFolderPath, 'train'))
    devSamples = loadRaceSplit(os.path.join(raceFolderPath, 'dev'))
    print(f'RACE train size: {len(trainSamples)}')
    print(f'RACE validation size: {len(devSamples)}')
    return {'train': trainSamples, 'validation': devSamples}

raceData = loadRaceLocal(raceFolderPath)

RACE train size: 87866
RACE validation size: 4887


## Step 6: Explore Dataset Samples

In [26]:
# Print one sample from SQuAD validation set
def showSquadSample(squadData, index=0):
    # Prints context snippet, question, and first answer
    sample = squadData['validation'][index]
    print('Context:', sample['context'][:300])
    print('Question:', sample['question'])
    print('Answer:', sample['answers'][0]['text'])

# Print one sample from RACE validation set
def showRaceSample(raceData, index=0):
    # Prints article snippet, question, options, and correct answer
    sample = raceData['validation'][index]
    print('Article:', sample['article'][:300])
    print('Question:', sample['question'])
    print('Options:', sample['options'])
    print('Answer:', sample['answer'])

print('--- SQuAD Sample ---')
showSquadSample(squadData)
print('\n--- RACE Sample ---')
showRaceSample(raceData)

--- SQuAD Sample ---
Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super B
Question: Which NFL team represented the AFC at Super Bowl 50?
Answer: Denver Broncos

--- RACE Sample ---
Article: Asian athletes have had a spare time in the first two days of the World Indoor Championships in Birmingham, England. But Chinese hurdler  Liu Xiang surprised everyone by taking the bronze medal in the men's 60-meter hurdles.
Liu became the first Chinese male athlete to get a world indoor medal in th
Question: The first sentence "Asian athletes have had a spare time in the first two days..." means "  _  ".
Options: ['Asian athletes can do things at their will in the first two days', "Asian athletes haven't any achievements in the first two days", '

## Step 7: Key Concept Extraction using NER

In [27]:
# Extract named entities from text using spacy
def extractKeyEntities(text):
    # Returns list of entity strings found in the text
    doc = nlpModel(text)
    return [ent.text for ent in doc.ents]

# Extract noun chunks from text
def extractNounChunks(text):
    # Returns list of noun chunk strings
    doc = nlpModel(text)
    return [chunk.text for chunk in doc.noun_chunks]

# Combine entities and noun chunks into a deduplicated concept list
def extractKeyConcepts(text):
    # Returns unique concepts from NER and noun chunking
    entities = extractKeyEntities(text)
    chunks = extractNounChunks(text)
    return list(set(entities + chunks))

# Test concept extraction on a SQuAD sample
sampleText = squadData['train'][0]['context']
keyConcepts = extractKeyConcepts(sampleText[:500])
print('Extracted key concepts:', keyConcepts[:10])

Extracted key concepts: ['Venite Ad Me Omnes', 'reflection', 'prayer', "the Main Building's gold dome", 'a copper statue', 'a replica', 'Catholic', 'Christ', 'France', 'arms']


## Step 8: Load T5 Model

In [28]:
# Load the T5 tokenizer and model onto the configured device
def loadT5Model(modelName='t5-small'):
    # Returns (tokenizer, model) tuple
    tokenizer = T5Tokenizer.from_pretrained(modelName)
    model = T5ForConditionalGeneration.from_pretrained(modelName)
    model = model.to(device)
    return tokenizer, model

t5Tokenizer, t5Model = loadT5Model('t5-small')
print('T5 model loaded successfully')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5 model loaded successfully


## Step 9: Prepare SQuAD Samples for T5

In [29]:
# Format one SQuAD sample as a T5 input/target pair
def formatSquadSample(sample):
    # Returns dict with inputText and targetText keys
    context = sample['context']
    answer = sample['answers'][0]['text']
    question = sample['question']
    inputText = f'generate question: context: {context} answer: {answer}'
    return {'inputText': inputText, 'targetText': question}

# Prepare a batch of formatted samples from one split
def prepareSquadBatch(squadData, split='train', batchSize=500):
    # Returns a list of formatted input/output dicts
    formattedSamples = []
    limit = min(batchSize, len(squadData[split]))
    for i in range(limit):
        formattedSamples.append(formatSquadSample(squadData[split][i]))
    return formattedSamples

trainSamples = prepareSquadBatch(squadData, split='train', batchSize=5000)
valSamples = prepareSquadBatch(squadData, split='validation', batchSize=5000)
print(f'Prepared {len(trainSamples)} train samples and {len(valSamples)} val samples')

Prepared 5000 train samples and 5000 val samples


## Step 10: Dataset Class and DataLoaders

In [30]:
from torch.utils.data import Dataset, DataLoader

# Tokenize one formatted sample for T5
def tokenizeSample(sample, tokenizer, maxInputLen, maxTargetLen):
    # Returns dict with input_ids, attention_mask, and labels tensors
    inputEncoded = tokenizer(
        sample['inputText'], max_length=maxInputLen,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    targetEncoded = tokenizer(
        sample['targetText'], max_length=maxTargetLen,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    labels = targetEncoded['input_ids'].squeeze()
    labels[labels == tokenizer.pad_token_id] = -100
    return {
        'input_ids': inputEncoded['input_ids'].squeeze(),
        'attention_mask': inputEncoded['attention_mask'].squeeze(),
        'labels': labels
    }

# Custom PyTorch dataset wrapping formatted samples
class QuestionDataset(Dataset):
    # Wraps samples list for use with DataLoader
    def __init__(self, samples, tokenizer, maxInputLen=512, maxTargetLen=64):
        self.samples = samples
        self.tokenizer = tokenizer
        self.maxInputLen = maxInputLen
        self.maxTargetLen = maxTargetLen

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return tokenizeSample(
            self.samples[idx], self.tokenizer,
            self.maxInputLen, self.maxTargetLen
        )

trainDataset = QuestionDataset(trainSamples, t5Tokenizer)
valDataset = QuestionDataset(valSamples, t5Tokenizer)
trainLoader = DataLoader(trainDataset, batch_size=8, shuffle=True)
valLoader = DataLoader(valDataset, batch_size=8, shuffle=False)
print(f'Train batches: {len(trainLoader)}, Val batches: {len(valLoader)}')

Train batches: 625, Val batches: 625


## Step 11: Fine-Tune T5

In [33]:
from torch.optim import AdamW

# Run a single training epoch and return average loss
def trainOneEpoch(model, dataLoader, optimizer):
    # Returns average loss across all batches
    model.train()
    totalLoss = 0
    for batch in dataLoader:
        inputIds = batch['input_ids'].to(device)
        attentionMask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        output = model(input_ids=inputIds, attention_mask=attentionMask, labels=labels)
        output.loss.backward()
        optimizer.step()
        totalLoss += output.loss.item()
    return totalLoss / len(dataLoader)

# Evaluate model on validation loader and return average loss
def evaluateModel(model, dataLoader):
    # Returns average validation loss without updating weights
    model.eval()
    totalLoss = 0
    with torch.no_grad():
        for batch in dataLoader:
            inputIds = batch['input_ids'].to(device)
            attentionMask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            output = model(input_ids=inputIds, attention_mask=attentionMask, labels=labels)
            totalLoss += output.loss.item()
    return totalLoss / len(dataLoader)

# Run full training loop for given number of epochs
def runTraining(model, trainLoader, valLoader, numEpochs=8, learningRate=5e-5):
    # Prints train and val loss after each epoch
    optimizer = AdamW(model.parameters(), lr=learningRate)
    for epoch in range(numEpochs):
        trainLoss = trainOneEpoch(model, trainLoader, optimizer)
        valLoss = evaluateModel(model, valLoader)
        print(f'Epoch {epoch+1}/{numEpochs} | Train Loss: {trainLoss:.4f} | Val Loss: {valLoss:.4f}')

runTraining(t5Model, trainLoader, valLoader, numEpochs=8)

Epoch 1/8 | Train Loss: 1.4272 | Val Loss: 1.9707
Epoch 2/8 | Train Loss: 1.3866 | Val Loss: 1.9779
Epoch 3/8 | Train Loss: 1.3483 | Val Loss: 1.9943
Epoch 4/8 | Train Loss: 1.3083 | Val Loss: 2.0031
Epoch 5/8 | Train Loss: 1.2752 | Val Loss: 2.0181
Epoch 6/8 | Train Loss: 1.2437 | Val Loss: 2.0172
Epoch 7/8 | Train Loss: 1.2036 | Val Loss: 2.0468
Epoch 8/8 | Train Loss: 1.1798 | Val Loss: 2.0561


## Step 12: Save Fine-Tuned Model

In [34]:
# Save model weights and tokenizer to disk
def saveModel(model, tokenizer, savePath):
    # Writes model and tokenizer files to savePath directory
    model.save_pretrained(savePath)
    tokenizer.save_pretrained(savePath)
    print(f'Model saved to {savePath}')

saveModel(t5Model, t5Tokenizer, './exam_gen_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./exam_gen_model


## Step 13: Generate a Question from Context and Answer

In [35]:
# Use T5 to generate a question given a context passage and an answer
def generateQuestion(model, tokenizer, context, answer, maxLen=64):
    # Returns the generated question as a string
    inputText = f'generate question: context: {context} answer: {answer}'
    inputEncoded = tokenizer(
        inputText, return_tensors='pt', max_length=512, truncation=True
    ).to(device)
    outputIds = model.generate(
        input_ids=inputEncoded['input_ids'],
        attention_mask=inputEncoded['attention_mask'],
        max_length=maxLen,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputIds[0], skip_special_tokens=True)

# Test on one SQuAD validation sample
testSample = squadData['validation'][5]
testAnswer = testSample['answers'][0]['text']
generatedQuestion = generateQuestion(t5Model, t5Tokenizer, testSample['context'], testAnswer)
print('Answer:', testAnswer)
print('Generated Question:', generatedQuestion)
print('Ground Truth:', testSample['question'])

Answer: "golden anniversary"
Generated Question: What did the national football league emphasize?
Ground Truth: What was the theme of Super Bowl 50?


## Step 14: MCQ Generation from RACE

In [36]:
# Map a RACE answer letter (A/B/C/D) to the option text
def resolveRaceAnswer(sample):
    # Returns the correct option string for the sample
    optionLabels = ['A', 'B', 'C', 'D']
    correctIndex = optionLabels.index(sample['answer'])
    return sample['options'][correctIndex]

# Build a structured MCQ dict from one RACE sample
def buildMcqFromRaceSample(sample):
    # Returns dict with article snippet, question, options, and correct answer
    return {
        'article': sample['article'][:300],
        'question': sample['question'],
        'options': sample['options'],
        'correctAnswer': resolveRaceAnswer(sample)
    }

# Generate a list of MCQ dicts from RACE validation split
def generateMcqs(raceData, count=5):
    # Returns a list of MCQ dicts
    return [buildMcqFromRaceSample(raceData['validation'][i]) for i in range(count)]

mcqList = generateMcqs(raceData, count=5)
for i, mcq in enumerate(mcqList):
    print(f'\nMCQ {i+1}: {mcq["question"]}')
    for j, opt in enumerate(mcq['options']):
        print(f'  {chr(65+j)}) {opt}')
    print(f'  Correct: {mcq["correctAnswer"]}')


MCQ 1: The first sentence "Asian athletes have had a spare time in the first two days..." means "  _  ".
  A) Asian athletes can do things at their will in the first two days
  B) Asian athletes haven't any achievements in the first two days
  C) Asian athletes could match athletes from other continents
  D) Asian athletes are tired of competing in the first two days
  Correct: Asian athletes haven't any achievements in the first two days

MCQ 2: Liu   _   the World Championship outdoors in Paris later this year.
  A) is busy preparing for
  B) has great pressure on
  C) is more confident of his ability in
  D) pays little attention to
  Correct: is more confident of his ability in

MCQ 3: Which of the following is TRUE according to the passage?
  A) Liu Xiang was the youngest athlete to take part in the competition in England.
  B) Liu Xiang was the first Asian medallist to get a world indoor medal.
  C) Liu Xiang is not only a good athlete but a music-lover.
  D) Liu Xiang finds it 

## Step 15: Short Answer Question Generation from SQuAD

In [37]:
# Generate short-answer questions using SQuAD context and answer pairs
def generateShortAnswerQuestions(squadData, model, tokenizer, count=5):
    # Returns list of dicts with generated question and expected answer
    resultList = []
    for i in range(count):
        sample = squadData['validation'][i]
        context = sample['context']
        answer = sample['answers'][0]['text']
        question = generateQuestion(model, tokenizer, context, answer)
        resultList.append({'question': question, 'answer': answer})
    return resultList

shortAnswers = generateShortAnswerQuestions(squadData, t5Model, t5Tokenizer, count=5)
for i, qa in enumerate(shortAnswers):
    print(f'\nQ{i+1}: {qa["question"]}')
    print(f'  Answer: {qa["answer"]}')


Q1: Who defeated Carolina Panthers 24–10 in Super Bowl 50?
  Answer: Denver Broncos

Q2: Who defeated the National Football Conference champion?
  Answer: Carolina Panthers

Q3: Where was the Super Bowl played?
  Answer: Santa Clara, California

Q4: Who defeated Carolina Panthers 24–10 in Super Bowl 50?
  Answer: Denver Broncos

Q5: What was the name of the 50th Super Bowl?
  Answer: gold


## Step 16: Long Descriptive Question Generation from RACE

In [90]:
def loadDescriptiveModel():
    # Returns tokenizer and model for descriptive question generation
    descTokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
    descModel = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')
    descModel = descModel.to(device)
    return descTokenizer, descModel
 
descTokenizer, descModel = loadDescriptiveModel()
print('Descriptive model loaded')
 
 
# Score a sentence by richness - more entities and chunks = more important
def scoreSentence(sentence):
    # Returns count of named entities and noun chunks as importance score
    doc = nlpModel(sentence)
    return len(doc.ents) + len(list(doc.noun_chunks))
 
 
# Pick the most content-rich unique sentences from the passage
def selectRichSentences(passage, count):
    # Returns top sentences ranked by NER and noun chunk density
    sentences = [s.strip() for s in passage.split('.') if len(s.strip()) > 30]
    scored = [(s, scoreSentence(s)) for s in sentences]
    scored.sort(key=lambda x: x[1], reverse=True)
    seenSentences = set()
    uniqueSentences = []
    for s, score in scored:
        key = s[:60].lower()
        if key not in seenSentences:
            seenSentences.add(key)
            uniqueSentences.append(s)
        if len(uniqueSentences) >= count * 2:
            break
    return uniqueSentences
 
 
# Generate one descriptive question using flan-t5-base with full context
def generateDescriptiveQuestion(descModel, descTokenizer, context, sentence):
    # Prompts flan-t5 to write a deep descriptive question about the sentence
    prompt = (
        f'Read the following passage and write one detailed descriptive question '
        f'about this key event or idea: "{sentence}". '
        f'The question should require a paragraph-length answer and begin with '
        f'words like How, Why, Describe, Explain, or Discuss.\n\n'
        f'Passage: {context[:600]}'
    )
    inputEncoded = descTokenizer(
        prompt, return_tensors='pt', max_length=600,
        truncation=True
    ).to(device)
    outputIds = descModel.generate(
        input_ids=inputEncoded['input_ids'],
        attention_mask=inputEncoded['attention_mask'],
        max_length=80,
        num_beams=5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    return descTokenizer.decode(outputIds[0], skip_special_tokens=True)
 
 
# Generate descriptive questions for multiple unique RACE articles
def generateDescriptiveQuestions(raceData, descModel, descTokenizer, count=3):
    # Deduplicates articles and generates one rich question per unique article
    seenArticles = set()
    seenQuestions = set()
    resultList = []
    index = 0
    validationData = raceData['validation']
 
    while len(resultList) < count and index < len(validationData):
        article = validationData[index]['article']
        articleKey = article[:100]
 
        if articleKey not in seenArticles:
            seenArticles.add(articleKey)
            richSentences = selectRichSentences(article, count=count)
 
            for sentence in richSentences:
                if len(resultList) >= count:
                    break
                question = generateDescriptiveQuestion(descModel, descTokenizer, article, sentence)
                questionKey = question.strip().lower()
                if questionKey not in seenQuestions:
                    seenQuestions.add(questionKey)
                    resultList.append({
                        'question': question,
                        'articleSnippet': article[:200]
                    })
 
        index += 1
 
    return resultList
 
 
descriptiveQuestions = generateDescriptiveQuestions(raceData, descModel, descTokenizer, count=3)
for i, item in enumerate(descriptiveQuestions):
    print(f'\nDescriptive Q{i+1}: {item["question"]}')
    print(f'  Based on: {item["articleSnippet"]}...')
 
 

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Descriptive model loaded

Descriptive Q1: How did Liu Xiang surprise everyone by taking the bronze medal in the 60-meter hurdles?
  Based on: Asian athletes have had a spare time in the first two days of the World Indoor Championships in Birmingham, England. But Chinese hurdler  Liu Xiang surprised everyone by taking the bronze medal in the...

Descriptive Q2: How did Liu Xiang react?
  Based on: Asian athletes have had a spare time in the first two days of the World Indoor Championships in Birmingham, England. But Chinese hurdler  Liu Xiang surprised everyone by taking the bronze medal in the...

Descriptive Q3: Describe, Explain, or Describe
  Based on: Asian athletes have had a spare time in the first two days of the World Indoor Championships in Birmingham, England. But Chinese hurdler  Liu Xiang surprised everyone by taking the bronze medal in the...


## Step 17: Assemble Balanced Exam Paper

In [82]:
# Combine all question types into one balanced exam paper
def generateExamPaper(squadData, raceData, model, tokenizer, mcqCount=5, shortCount=5, descCount=3):
    # Returns a dict containing mcqs, shortAnswerQuestions, and descriptiveQuestions
    print('Generating MCQs from RACE...')
    mcqs = generateMcqs(raceData, count=mcqCount)

    print('Generating short answer questions from SQuAD...')
    shortAnswers = generateShortAnswerQuestions(squadData, model, tokenizer, count=shortCount)

    print('Generating descriptive questions from RACE...')
    descriptive = generateDescriptiveQuestions(raceData, model, tokenizer, count=descCount)

    return {
        'mcqs': mcqs,
        'shortAnswerQuestions': shortAnswers,
        'descriptiveQuestions': descriptive
    }

examPaper = generateExamPaper(squadData, raceData, t5Model, t5Tokenizer)
print(f'\nGenerated: {len(examPaper["mcqs"])} MCQs, {len(examPaper["shortAnswerQuestions"])} Short Answers, {len(examPaper["descriptiveQuestions"])} Descriptive')

Generating MCQs from RACE...
Generating short answer questions from SQuAD...
Generating descriptive questions from RACE...

Generated: 5 MCQs, 5 Short Answers, 3 Descriptive


## Step 18: Save Exam Paper to JSON

In [83]:
# Write the exam paper dict to a JSON file
def saveExamPaper(examPaper, filePath='exam_paper.json'):
    # Saves exam paper to filePath as formatted JSON
    with open(filePath, 'w', encoding='utf-8') as outputFile:
        json.dump(examPaper, outputFile, indent=2)
    print(f'Exam paper saved to {filePath}')

saveExamPaper(examPaper)

Exam paper saved to exam_paper.json


## Step 19: Evaluate with ROUGE-L Score

In [85]:
# Compute ROUGE-L F1 score between a generated and reference question
def computeRougeScore(generatedQuestion, referenceQuestion):
    # Returns ROUGE-L F1 score as a float
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = scorer.score(referenceQuestion, generatedQuestion)
    return scores['rougeL'].fmeasure

# Evaluate on samples from unique contexts to get topic diversity
def evaluateGeneratedQuestions(squadData, model, tokenizer, count=10):
    # Prints per-sample and average ROUGE-L scores across unique articles
    seenContexts = set()
    resultList = []
    index = 0
    validationData = squadData['validation']

    while len(resultList) < count and index < len(validationData):
        sample = validationData[index]
        contextKey = sample['context'][:100]

        if contextKey not in seenContexts:
            seenContexts.add(contextKey)
            answer = sample['answers'][0]['text']
            referenceQuestion = sample['question']
            generatedQuestion = generateQuestion(model, tokenizer, sample['context'], answer)
            score = computeRougeScore(generatedQuestion, referenceQuestion)
            resultList.append(score)
            print(f'Sample {len(resultList)} | ROUGE-L: {score:.4f} | Generated: {generatedQuestion}')

        index += 1

    avgScore = sum(resultList) / len(resultList)
    print(f'\nAverage ROUGE-L: {avgScore:.4f}')

evaluateGeneratedQuestions(squadData, t5Model, t5Tokenizer, count=10)

Sample 1 | ROUGE-L: 0.3000 | Generated: Who defeated Carolina Panthers 24–10 in Super Bowl 50?
Sample 2 | ROUGE-L: 0.5000 | Generated: Who was the NFL Most Valuable Player?
Sample 3 | ROUGE-L: 0.5333 | Generated: Which Denver linebacker was named Super Bowl MVP?
Sample 4 | ROUGE-L: 0.7368 | Generated: Which company broadcasts Super Bowl 50 in the US?
Sample 5 | ROUGE-L: 0.2105 | Generated: Which NFL commissioner stated that the 50th Super Bowl was "spectacular"?
Sample 6 | ROUGE-L: 0.0000 | Generated: What site did the league narrow its bids to?
Sample 7 | ROUGE-L: 0.2963 | Generated: When did the league announce the two finalists were Sun Life Stadium and Levi's Stadium?
Sample 8 | ROUGE-L: 0.3636 | Generated: When did NFL owners vote for Levi's Stadium?
Sample 9 | ROUGE-L: 0.4828 | Generated: Who coached the Panthers in their last Super Bowl appearance?
Sample 10 | ROUGE-L: 0.2500 | Generated: Which longtime running back did the Carolina Panthers lose?

Average ROUGE-L: 0.3673


## Step 20: Inference on Custom Text

In [86]:
# Generate questions from any custom passage by extracting key concepts first
WEAK_STARTERS = {'the', 'a', 'an', 'this', 'that', 'these', 'those', 'one', 'many'}


def isValidConcept(concept):
    # Returns True only if concept has 2 or more real words and is not just a number
    words = [word for word in concept.split() if word.isalpha()]
    firstWord = words[0].lower() if words else ''
    return len(words) >= 2 and firstWord not in WEAK_STARTERS

def generateQuestionsFromCustomText(model, tokenizer, customText, numQuestions=3):
    # Returns list of dicts with concept and generated question
    keyConcepts = extractKeyConcepts(customText)[:numQuestions]
    resultList = []
    for concept in keyConcepts:
        question = generateQuestion(model, tokenizer, customText, concept)
        resultList.append({'concept': concept, 'question': question})
    return resultList

# Test with a sample biology passage
customPassage = """
The Great Fire swept across London in 1666, originating in a baker's shop, and initially seemed insignificant. However, the weather had been hot and dry for some time, and the area where the fire started was filled with old wooden buildings situated very close to each other. Additionally, a strong wind carried burning pieces of wood to the roofs of distant houses, causing the fire to spread rapidly. Primitive firefighting techniques, such as lines of men passing buckets of water, proved ineffective. When the wind ceased, the fire was eventually extinguished. By the time the fire was out, four-fifths of the city had been destroyed, and nearly a quarter of a million people had lost their homes.
"""

customQuestions = generateQuestionsFromCustomText(t5Model, t5Tokenizer, customPassage, numQuestions=3)
for item in customQuestions:
    print(f'Concept: {item["concept"]}')
    print(f'Question: {item["question"]}\n')

Concept: a million people
Question: How many people lost their homes in the Great Fire?

Concept: the roofs
Question: What did the wind carry burning pieces of wood to?

Concept: London
Question: Where was the Great Fire in 1666?



# GENERATE Q FROM USER REQUEST :

In [87]:
# Ask user for input passage and question counts interactively
def getUserInputs():
    # Returns the passage text and question counts entered by the user
    print('Enter your passage below. When done, type END on a new line and press Enter:')
    lines = []
    while True:
        line = input()
        if line.strip() == 'END':
            break
        lines.append(line)
    userPassage = '\n'.join(lines)

    shortCount = int(input('How many short answer questions do you want? '))
    descCount = int(input('How many descriptive questions do you want? '))

    return userPassage, shortCount, descCount

In [72]:
# Generate MCQs from user passage using concept extraction and distractors from RACE

# Generate short answers from second half of concept pool
def generateShortAnswersFromPassage(model, tokenizer, passage, count):
    # Uses second half of valid concepts to avoid overlap with MCQ section
    allConcepts = extractKeyConcepts(passage)
    validConcepts = [c for c in allConcepts if isValidConcept(c)]
    halfPoint = len(validConcepts) // 2
    shortConcepts = validConcepts[halfPoint:]

    seenQuestions = set()
    resultList = []

    for concept in shortConcepts:
        if len(resultList) >= count:
            break
        question = generateQuestion(model, tokenizer, passage, concept)
        questionKey = question.strip().lower()
        if questionKey not in seenQuestions:
            seenQuestions.add(questionKey)
            resultList.append({'question': question, 'answer': concept})

    return resultList
# Extract short noun phrases from a sentence to use as distractors
def extractPhrasesFromSentence(sentence):
    # Returns list of noun chunk strings from one sentence
    doc = nlpModel(sentence)
    phrases = [chunk.text for chunk in doc.noun_chunks if isValidConcept(chunk.text)]
    return phrases

# Build distractors from sentences that contain the correct answer
def pickDistractors(passage, correctAnswer, allConcepts, numDistractors=3):
    # Finds sentences containing the answer and pulls sibling phrases as distractors
    sentences = [s.strip() for s in passage.split('.') if len(s.strip()) > 10]
    distractors = []
    seen = set()
    seen.add(correctAnswer.lower())

    # First pass: find phrases from sentences that contain the correct answer
    for sentence in sentences:
        if correctAnswer.lower() in sentence.lower():
            phrases = extractPhrasesFromSentence(sentence)
            for phrase in phrases:
                if len(distractors) >= numDistractors:
                    break
                if phrase.lower() not in seen:
                    distractors.append(phrase)
                    seen.add(phrase.lower())

    # Second pass: fill remaining slots from other sentences if not enough found
    if len(distractors) < numDistractors:
        for sentence in sentences:
            if len(distractors) >= numDistractors:
                break
            phrases = extractPhrasesFromSentence(sentence)
            for phrase in phrases:
                if len(distractors) >= numDistractors:
                    break
                if phrase.lower() not in seen:
                    distractors.append(phrase)
                    seen.add(phrase.lower())

    return distractors[:numDistractors]

In [91]:
def generateDescriptiveFromPassage(descModel, descTokenizer, passage, count, existingQuestions):
    # Selects richest sentences and generates open-ended questions via flan-t5
    richSentences = selectRichSentences(passage, count=count)
    seenQuestions = set(q.strip().lower() for q in existingQuestions)
    seenSentenceKeys = set()
    resultList = []
 
    for sentence in richSentences:
        if len(resultList) >= count:
            break
 
        sentenceKey = sentence[:60].lower()
        if sentenceKey in seenSentenceKeys:
            continue
        seenSentenceKeys.add(sentenceKey)
 
        question = generateDescriptiveQuestion(descModel, descTokenizer, passage, sentence)
        questionKey = question.strip().lower()
 
        if questionKey not in seenQuestions:
            seenQuestions.add(questionKey)
            resultList.append({'question': question, 'hint': sentence})
 
    return resultList
 

# WITH UI 

In [92]:
# Count maximum unique questions generatable from a passage
def countMaxUniqueQuestions(passage):
    # Returns max short answer and descriptive counts based on passage content
    allConcepts = extractKeyConcepts(passage)
    validConcepts = [c for c in allConcepts if isValidConcept(c)]
    sentences = [s.strip() for s in passage.split('.') if len(s.strip()) > 30]
    maxShort = len(validConcepts)
    maxDesc = len(sentences)
    return maxShort, maxDesc

# Ask user for question counts and validate against passage capacity
def getQuestionCounts(maxShort, maxDesc):
    # Returns validated short and descriptive counts from user input
    print(f'\nThis passage can generate a maximum of:')
    print(f'   Short answer questions : {maxShort}')
    print(f'   Descriptive questions  : {maxDesc}')
    print()

    shortCount = int(input('How many short answer questions do you want? '))
    descCount = int(input('How many descriptive questions do you want? '))

    # Clamp to max if user asks for more than passage can support
    if shortCount > maxShort:
        print(f'Only {maxShort} unique short answer questions possible. Setting to {maxShort}.')
        shortCount = maxShort
    if descCount > maxDesc:
        print(f'Only {maxDesc} unique descriptive questions possible. Setting to {maxDesc}.')
        descCount = maxDesc

    return shortCount, descCount

# Get passage from user via multiline input
def getUserPassage():
    # Returns the passage string entered by the user
    print('Enter your passage below. When done, type END on a new line and press Enter:')
    lines = []
    while True:
        line = input()
        if line.strip() == 'END':
            break
        lines.append(line)
    return '\n'.join(lines)

def printExamPaper(shortAnswers, descriptiveQuestions):
    # Prints short answer and descriptive sections with numbering
    print('\n' + '='*60)
    print('SECTION A - SHORT ANSWER QUESTIONS')
    print('='*60)
    for i, qa in enumerate(shortAnswers):
        print(f'\nQ{i+1}. {qa["question"]}')
        print(f'   [Answer: {qa["answer"]}]')

    print('\n' + '='*60)
    print('SECTION B - DESCRIPTIVE QUESTIONS')
    print('='*60)
    for i, item in enumerate(descriptiveQuestions):
        print(f'\nQ{i+1}. {item["question"]}')
# Run the full interactive exam paper generator with capacity check
def runExamPaperGenerator(model, tokenizer):
    # Collects passage, shows max capacity, validates counts, generates paper
    userPassage = getUserPassage()

    maxShort, maxDesc = countMaxUniqueQuestions(userPassage)

    # Warn user if passage is too short to generate meaningful questions
    if maxShort == 0 and maxDesc == 0:
        print('\nPassage is too short to generate any questions. Please provide a longer passage.')
        return {}

    shortCount, descCount = getQuestionCounts(maxShort, maxDesc)

    print('\nGenerating your exam paper, please wait...')

    shortAnswers = generateShortAnswersFromPassage(model, tokenizer, userPassage, shortCount)

    existingQuestions = [qa['question'] for qa in shortAnswers]

    descriptive = generateDescriptiveFromPassage(
        model, tokenizer, userPassage, descCount, existingQuestions
    )

    printExamPaper(shortAnswers, descriptive)

    return {
        'shortAnswerQuestions': shortAnswers,
        'descriptiveQuestions': descriptive
    }

examPaper = runExamPaperGenerator(t5Model, t5Tokenizer)

Enter your passage below. When done, type END on a new line and press Enter:


 END



Passage is too short to generate any questions. Please provide a longer passage.


In [93]:
# ============================================================
# TEXT FILE INPUT - Upload a .txt file and generate questions
# ============================================================
# This cell lets you upload a .txt file instead of typing a passage.
# After running this cell, use the file picker that appears to select your file.

import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# --- Widget UI ---
upload_widget = widgets.FileUpload(
    accept='.txt',
    multiple=False,
    description='Upload .txt file',
    layout=widgets.Layout(width='300px')
)

short_count_widget = widgets.BoundedIntText(
    value=5,
    min=1,
    max=50,
    description='Short Qs:',
    layout=widgets.Layout(width='200px')
)

desc_count_widget = widgets.BoundedIntText(
    value=3,
    min=1,
    max=20,
    description='Descriptive Qs:',
    layout=widgets.Layout(width='200px')
)

generate_button = widgets.Button(
    description='Generate Exam Paper',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

output_area = widgets.Output()

def on_generate_clicked(b):
    with output_area:
        clear_output()
        
        # Read uploaded file
        if not upload_widget.value:
            print('Please upload a .txt file first.')
            return
        
        uploaded_file = upload_widget.value[0]
        file_content = bytes(uploaded_file['content'])  # convert memoryview → bytes

        try:
            user_passage = file_content.decode('utf-8')
        except UnicodeDecodeError:
            user_passage = file_content.decode('latin-1')
        
        print(f'File loaded: {len(user_passage)} characters, {len(user_passage.split())} words')
        print(f'Preview: {user_passage[:200]}...\n')
        
        # Check passage capacity
        maxShort, maxDesc = countMaxUniqueQuestions(user_passage)
        print(f'This passage can generate a maximum of:')
        print(f'  Short answer questions : {maxShort}')
        print(f'  Descriptive questions  : {maxDesc}\n')
        
        if maxShort == 0 and maxDesc == 0:
            print('Passage is too short to generate any questions. Please provide a longer .txt file.')
            return
        
        # Clamp counts to max
        short_count = min(short_count_widget.value, maxShort)
        desc_count = min(desc_count_widget.value, maxDesc)
        
        if short_count < short_count_widget.value:
            print(f'Note: Only {maxShort} unique short answer questions possible. Generating {maxShort}.')
        if desc_count < desc_count_widget.value:
            print(f'Note: Only {maxDesc} unique descriptive questions possible. Generating {maxDesc}.')
        
        print('Generating your exam paper, please wait...\n')
        
        shortAnswers = generateShortAnswersFromPassage(t5Model, t5Tokenizer, user_passage, short_count)
        existingQuestions = [qa['question'] for qa in shortAnswers]
        descriptive = generateDescriptiveFromPassage(t5Model, t5Tokenizer, user_passage, desc_count, existingQuestions)
        
        printExamPaper(shortAnswers, descriptive)

generate_button.on_click(on_generate_clicked)

# Layout
print('Upload a .txt file and set the number of questions, then click Generate.')
display(
    widgets.VBox([
        upload_widget,
        widgets.HBox([short_count_widget, desc_count_widget]),
        generate_button,
        output_area
    ])
)


Upload a .txt file and set the number of questions, then click Generate.
